## Golden Model for NMS Algorithm

Uses File I/O Golden Model for hardware verification inspired from this [article](https://thedatabus.in/python-iverilog-verification/)

The input were considering has these parameters
- The confidence score `c`
- Lower left cordinate `(x, y)`
- Upper right coordinate `(a, b)`

In [ ]:
x1, y1, a1, b1, c1 = 200, 300, 400, 500, 0.86  # point 1
x2, y2, a2, b2, c2 = 350, 350, 450, 600, 0.65  # point 2

In [ ]:
# Calculations needed for IoU

area1 = (a1 - x1) * (b1 - y1)
area2 = (a2 - x2) * (b2 - y2)

# maximize the lower left and minimize the upper right (nature of intersection)
xx = max(x1, x2)
yy = max(y1, y2)
aa = min(a1, a2)
bb = min(b1, b2)

# width and height
w = max(0, (aa - xx))
h = max(0, (bb - yy))

# intersection and union area
intersection_area = w * h
union_area = area1 + area2 - intersection_area
iou = intersection_area / union_area

In [ ]:
iou

In [ ]:
bbox1 = 200, 300, 400, 500, 0.85
bbox2 = 350, 350, 450, 600, 0.65

In [ ]:
def calculate_iou(bbox1: tuple, bbox2: tuple) -> float:
    """Calculate IoU value from bounding boxes.

    Assumes each bounding box is a `tuple` follows this structure
    `(lower left cordinate, upper right cordinate, confidence_score).

    Args:
        bbox1 (tuple): bounding box 1
        bbox2 (tuple): bounding box 2

    Returns:
        float: calulated IoU value
    """
    x1, y1, a1, b1, _ = bbox1
    x2, y2, a2, b2, _ = bbox2

    # Calculations needed for IoU
    area1 = (a1 - x1) * (b1 - y1)
    area2 = (a2 - x2) * (b2 - y2)

    # maximize the lower left and minimize the upper right (nature of intersection)
    xx = max(x1, x2)
    yy = max(y1, y2)
    aa = min(a1, a2)
    bb = min(b1, b2)

    # width and height
    w = max(0, (aa - xx))
    h = max(0, (bb - yy))

    # intersection and union area
    intersection_area = w * h
    union_area = area1 + area2 - intersection_area
    iou = intersection_area / union_area

    return iou

In [ ]:
def nms(boxes: list, iou_threshold: float) -> list:

    sorted_boxes = sorted(boxes, key=lambda box: box[4], reverse=True)
    valid = [True] * len(sorted_boxes)
    keep = []

    for i in range(len(sorted_boxes)):
        if valid[i]:
            keep.append(sorted_boxes[i])
            valid[i] = False

            for j in range(i + 1, len(sorted_boxes)):
                if valid[j]:
                    iou = calculate_iou(sorted_boxes[i], sorted_boxes[j])
                    if iou >= iou_threshold:
                        valid[j] = False

    return keep


In [ ]:
# (x1, y1, x2, y2, confidence)
test_boxes = [
    # Cluster A — cat detection (~8 overlapping boxes)
    (10, 10, 50, 50, 0.95),
    (12, 11, 52, 51, 0.90),
    (9, 13, 48, 53, 0.85),
    (14, 9, 54, 49, 0.70),
    (11, 14, 51, 54, 0.65),
    (15, 12, 55, 52, 0.55),
    (8, 8, 46, 46, 0.40),
    (13, 15, 53, 55, 0.30),
    # Cluster B — dog detection (~8 overlapping boxes)
    (100, 100, 150, 150, 0.92),
    (102, 101, 152, 151, 0.88),
    (98, 103, 148, 153, 0.80),
    (104, 99, 154, 149, 0.72),
    (101, 105, 151, 155, 0.60),
    (103, 98, 153, 148, 0.50),
    (97, 102, 147, 152, 0.42),
    (105, 104, 155, 154, 0.28),
    # Cluster C — car detection (~8 overlapping boxes)
    (200, 50, 260, 100, 0.93),
    (202, 52, 262, 102, 0.87),
    (198, 48, 258, 98, 0.78),
    (204, 53, 264, 103, 0.68),
    (201, 47, 261, 97, 0.58),
    (199, 55, 259, 105, 0.48),
    (203, 49, 263, 99, 0.38),
    (197, 51, 257, 101, 0.25),
    # Cluster D — person detection (~5 overlapping boxes)
    (50, 200, 100, 280, 0.91),
    (52, 202, 102, 282, 0.82),
    (48, 198, 98, 278, 0.73),
    (54, 203, 104, 283, 0.62),
    (47, 199, 97, 279, 0.45),
    # Isolated boxes — should all survive
    (300, 300, 340, 340, 0.75),
    (0, 280, 30, 310, 0.35),
    (280, 0, 320, 40, 0.20),
]

In [ ]:
nms(test_boxes, 0.5)